# PCA + KNN Ablation

# Setup

In [ ]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

from preprocessing import clean_data, engineer_data
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [ ]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/pca_cnn.csv"

########## DEBUG ##########
DEBUG = True

########## MODEL ##########
INNER_CV = 5
OUTER_CV = 5
SCORING = 'neg_root_mean_squared_error'
PCA__N_COMPONENTS = list (range (25, 31, 2))
KNN__N_NEIGHBORS = list (range (50, 251, 50))

# Init & Pre-Processing

In [3]:
data = pd.read_csv (TRAIN_PATH)
labels, cleaned_data = clean_data (data)
feature_engineer = FunctionTransformer (engineer_data)

# Model

In [4]:
# Pipeline
pipeline = Pipeline ([('f_eng', feature_engineer),
                      ('scaler', StandardScaler ()), 
                      ('pca', PCA ()),
                      ('knn', KNeighborsRegressor (weights = 'distance'))])

# Grid Search Double CV
param_grid = {'pca__n_components': PCA__N_COMPONENTS,
              'knn__n_neighbors': KNN__N_NEIGHBORS}

gs = GridSearchCV (estimator = pipeline,
                   param_grid = param_grid,
                   scoring = SCORING,
                   cv = INNER_CV)

nested_scores = cross_val_score (gs,
                                 cleaned_data,
                                 labels,
                                 cv = OUTER_CV,
                                 scoring = SCORING,
                                 n_jobs = -1)

# Evaluate
print (f"Nested RMSE: {-nested_scores.mean ()}")
if (DEBUG):
    print (f"Fold RMSEs: {-nested_scores}")

Nested RMSE: 5.162749411339459
Fold RMSEs: [5.13884821 5.16990083 5.20155578 5.14791604 5.1555262 ]


# Predict

In [5]:
# Build final model with all training data
gs.fit (cleaned_data, labels)
print ("Best parameters:", gs.best_params_)
print ("Best inner CV score:", -gs.best_score_)

final_model = gs.best_estimator_

# Predict
test_data = pd.read_csv (TEST_PATH)
_, cleaned_test_data = clean_data (test_data)
predictions = final_model.predict (cleaned_test_data)

# Save
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (predictions) + 1),
                          'Milk_Yield_L': predictions})
out_data.to_csv (OUT_PATH, index = False)

Best parameters: {'knn__n_neighbors': 150, 'pca__n_components': 29}
Best inner CV score: 5.162206442609472
